# SQL Toolkit — Airtel Project

This notebook contains the nine **“In this project”** queries from the Airtel classroom runbook, in tool order. Run from top to bottom after placing `airtel.db` at the path used in the connection cell. Expected checkpoints are recorded in the markdown below each query.

In [13]:
import sqlite3
import pandas as pd
from IPython.display import display

# Connection cell follows the sqlite3 + pandas pattern from section 0.2.
# Assumes this notebook is in notebooks/ and airtel.db is in the project data/ folder.
conn = sqlite3.connect("../data/raw/airtel.db")


**Tool 1 — 1A.7: By complaint category, how many tickets are there, how many are still open, and how many took more than seven days to close?**

In [14]:
q_1a_7 = """
SELECT category,
       COUNT(*) AS tickets,
       SUM(is_resolved = 0) AS still_open,
       SUM(resolution_days > 7) AS slow
FROM complaints
GROUP BY category
ORDER BY tickets DESC;
"""
display(pd.read_sql_query(q_1a_7, conn))


,category,tickets,still_open,slow
0,Slow Data,8035,2413,2822
1,Dropped Call,6365,1928,2230
2,Network Coverage,4885,1489,1732
3,Billing,3738,1089,1321
4,Recharge Issue,2679,744,983
5,SIM/KYC,1139,358,401


**Expected checkpoint (1A.7):** 6 categories; 26,841 tickets total. The runbook reports Slow Data: 8,035 tickets, 2,413 open, 2,822 slow; Dropped Call: 6,365 tickets, 1,928 open, 2,230 slow; Network Coverage: 4,885 tickets, 1,489 open, 1,732 slow; Billing: 3,738 tickets, 1,089 open, 1,321 slow; Recharge Issue: 2,679 tickets, 744 open, 983 slow; SIM/KYC: 1,139 tickets, 358 open, 401 slow.

**Tool 2 — 1A.11: For every subscriber, how many days of usage are recorded, what is their average daily data use, and are they prepaid or postpaid?**

In [15]:
q_1a_11 = """
WITH u AS (
    SELECT customer_id,
           COUNT(*) AS days_active,
           ROUND(AVG(data_mb), 1) AS avg_mb
    FROM usage_daily
    GROUP BY customer_id
)
SELECT c.customer_id,
       c.connection_type,
       u.days_active,
       u.avg_mb
FROM customers AS c
LEFT JOIN u
    ON u.customer_id = c.customer_id;
"""
display(pd.read_sql_query(q_1a_11, conn))


,customer_id,connection_type,days_active,avg_mb
0,1,Prepaid,17.0,729.7
1,2,Prepaid,11.0,256.6
2,3,Postpaid,55.0,1868.9
3,4,Prepaid,40.0,218.9
4,5,Prepaid,39.0,1146.8
...,...,...,...,...
21995,21996,Postpaid,79.0,6610.9
21996,21997,Prepaid,101.0,1400.0
21997,21998,Prepaid,65.0,75.1
21998,21999,Prepaid,18.0,1154.0


**Expected checkpoint (1A.11):** 22,000 rows after the LEFT JOIN. The roll-up has 21,996 subscriber rows, so four subscribers have NULL usage values. The first five rows in the runbook are: (1, Prepaid, 17, 729.7), (2, Prepaid, 11, 256.6), (3, Postpaid, 55, 1868.9), (4, Prepaid, 40, 218.9), (5, Prepaid, 39, 1146.8).

**Tool 3 — 1A.15: How many days has each subscriber been with Airtel as of 30 June 2025?**

In [16]:
q_1a_15 = """
SELECT customer_id,
       activation_date,
       CAST(julianday('2025-06-30') - julianday(activation_date) AS INT)
           AS tenure_days
FROM customers
LIMIT 5;
"""
display(pd.read_sql_query(q_1a_15, conn))

,customer_id,activation_date,tenure_days
0,1,2023-04-22,800
1,2,2023-09-06,663
2,3,2023-02-09,872
3,4,2024-03-16,471
4,5,2022-12-10,933


**Expected checkpoint (1A.15):** The first five tenure values are customer 1: 800 days; customer 2: 663; customer 3: 872; customer 4: 471; customer 5: 933. The runbook reports average tenure 697.6 days, shortest 122, longest 1,276 across the full customer base.

**Tool 4 — 1A.19: Did average data use change month by month over the six-month usage window?**

In [17]:
q_1a_19 = """
SELECT strftime('%Y-%m', usage_date) AS month,
       COUNT(*) AS rows_,
       ROUND(AVG(data_mb), 1) AS avg_mb
FROM usage_daily
GROUP BY month
ORDER BY month;
"""
display(pd.read_sql_query(q_1a_19, conn))


,month,rows_,avg_mb
0,2025-01,242593,1460.8
1,2025-02,218198,1476.9
2,2025-03,234699,1485.6
3,2025-04,217330,1497.0
4,2025-05,215366,1767.9
5,2025-06,198553,1790.0


**Expected checkpoint (1A.19):** 6 rows. January: 242,593 rows, average 1,460.8 MB; February: 218,198, 1,476.9 MB; March: 234,699, 1,485.6 MB; April: 217,330, 1,497.0 MB; May: 215,366, 1,767.9 MB; June: 198,553, 1,790.0 MB. The runbook says to note the May/June jump as an open question rather than explain it at this stage.

**Tool 5 — 1A.23: How many subscribers have never raised a complaint, and what is their churn rate?**

In [18]:
q_1a_23 = """
SELECT COUNT(*) AS subs,
       ROUND(100.0 * AVG(c.is_churned), 2) AS churn
FROM customers AS c
LEFT JOIN complaints AS x
    ON x.customer_id = c.customer_id
WHERE x.customer_id IS NULL;
"""
display(pd.read_sql_query(q_1a_23, conn))

,subs,churn
0,6631,20.03


**Expected checkpoint (1A.23):** 6,631 subscribers who never complained; churn rate 20.03%. For comparison, the full base has 22,000 subscribers and 23.75% churn. Four subscribers have no usage_daily record.

**Tool 6 — 1A.27: How many subscribers have raised at least one complaint, and what is their churn rate?**

In [19]:
q_1a_27 = """
SELECT COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customers AS c
WHERE EXISTS (
    SELECT 1
    FROM complaints AS x
    WHERE x.customer_id = c.customer_id
);
"""
display(pd.read_sql_query(q_1a_27, conn))


,subs,churn
0,15369,25.36


**Expected checkpoint (1A.27):** EXISTS counts subscribers, not tickets: 15,369 subscribers with at least one complaint; churn rate 25.36%. A direct join returns 26,841 ticket rows and a different churn rate (27.55%). The 15,369 subscribers with complaints plus 6,631 without complaints equals the 22,000-customer base.


**Tool 7 — 1A.31: Does churn vary across four groups of subscribers ranked by total recharge spend?**

In [20]:
q_1a_31 = """
WITH r AS (
    SELECT customer_id,
           SUM(amount_inr) AS spend
    FROM recharges
    GROUP BY customer_id
),
n AS (
    SELECT customer_id,
           spend,
           NTILE(4) OVER (ORDER BY spend) AS qt
    FROM r
)
SELECT n.qt,
       COUNT(*) AS subs,
       MIN(n.spend) AS lowest,
       MAX(n.spend) AS highest,
       ROUND(100.0 * AVG(c.is_churned), 2) AS churn
FROM n
JOIN customers AS c
    ON c.customer_id = n.customer_id
GROUP BY n.qt
ORDER BY n.qt;
"""
display(pd.read_sql_query(q_1a_31, conn))


,qt,subs,lowest,highest,churn
0,1,5500,99,3553,34.85
1,2,5500,3553,6384,24.53
2,3,5500,6384,10819,20.65
3,4,5500,10819,71634,14.98


**Expected checkpoint (1A.31):** Four quartiles of 5,500 subscribers each. Churn rates: quartile 1 (lowest spend) 34.85%; quartile 2 24.53%; quartile 3 20.65%; quartile 4 (highest spend) 14.98%. The ₹3,553 boundary appears as both the top of quartile 1 and the bottom of quartile 2 because equal spend values can be split by NTILE.

**Tool 8 — 1A.35: How much cumulative recharge revenue comes from the biggest circles, ranked from highest revenue down?**


In [21]:
q_1a_35 = """
WITH r AS (
    SELECT ci.circle_name,
           SUM(rc.amount_inr) AS rev
    FROM recharges AS rc
    JOIN customers AS c
        ON c.customer_id = rc.customer_id
    JOIN circles AS ci
        ON ci.circle_id = c.circle_id
    GROUP BY ci.circle_name
)
SELECT circle_name,
       rev,
       SUM(rev) OVER (ORDER BY rev DESC) AS running,
       ROUND(
           100.0 * SUM(rev) OVER (ORDER BY rev DESC)
           / (SELECT SUM(rev) FROM r),
           1
       ) AS pct
FROM r
ORDER BY rev DESC;
"""
display(pd.read_sql_query(q_1a_35, conn))


,circle_name,rev,running,pct
0,Mumbai,13993855,13993855,6.4
1,Andhra Pradesh,13675963,27669818,12.7
2,Maharashtra,13432045,41101863,18.9
3,North East,13008584,54110447,24.9
4,Assam,12915233,67025680,30.8
5,Karnataka,12797614,79823294,36.7
6,Gujarat,12552746,92376040,42.5
7,Himachal Pradesh,11849346,104225386,48.0
8,Kerala,11562234,115787620,53.3
9,Uttar Pradesh (West),11307195,127094815,58.5


**Expected checkpoint (1A.35):** The first six circles together account for 36.7% of revenue. The first three rows in the runbook are Mumbai (₹13,993,855; cumulative ₹13,993,855; 6.4%), Andhra Pradesh (₹13,675,963; cumulative ₹27,669,818; 12.7%), and Maharashtra (₹13,432,045; cumulative ₹41,101,863; 18.9%).

**Tool 9 — 1A.39: Which indexes exist in airtel.db, and does SQLite use an index for subscriber lookup versus a data_mb filter?**


In [22]:
# 1) List indexes that already exist in airtel.db
q_1a_39_indexes = """
SELECT name, tbl_name
FROM sqlite_master
WHERE type = 'index'
ORDER BY tbl_name, name;
"""
display(pd.read_sql_query(q_1a_39_indexes, conn))

# 2) Explain the plan for a subscriber lookup on indexed customer_id
q_1a_39_customer = """
EXPLAIN QUERY PLAN
SELECT usage_date, data_mb
FROM usage_daily
WHERE customer_id = 9;
"""
display(pd.read_sql_query(q_1a_39_customer, conn))

# 3) Explain the plan for a data_mb filter without a listed index
q_1a_39_data = """
EXPLAIN QUERY PLAN
SELECT usage_date, data_mb
FROM usage_daily
WHERE data_mb > 5000;
"""
display(pd.read_sql_query(q_1a_39_data, conn))


,name,tbl_name
0,ix_c_cust,complaints
1,ix_cust_circle,customers
2,ix_cust_plan,customers
3,ix_r_cust,recharges
4,ix_u_cust,usage_daily
5,ix_u_date,usage_daily


,id,parent,notused,detail
0,3,0,61,SEARCH usage_daily USING INDEX ix_u_cust (cust...


,id,parent,notused,detail
0,2,0,216,SCAN usage_daily


**Expected checkpoint (1A.39):** The runbook lists six existing indexes: ix_u_cust on usage_daily, ix_u_date on usage_daily, ix_r_cust on recharges, ix_c_cust on complaints, ix_cust_circle on customers, and ix_cust_plan on customers. The customer_id lookup should show a SEARCH using ix_u_cust; the data_mb > 5000 query should show SCAN usage_daily. Exact EXPLAIN wording may vary by SQLite version.